# Cookmate AI Monitoring and LLM judge

We explore and set the data in the right format for mlflow judge

In [15]:
from pathlib import Path
import json
import pandas as pd
import lancedb
from sentence_transformers import SentenceTransformer


## Project Path

In [16]:
from cookmate.utils.config import (
    BASE_DIR,
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    DB_DIR,
    EVALUATION_DATASET_PATH,
    RECIPES_JSON_PATH,
)

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Database directory:", DB_DIR)
print("Evaluation dataset path:", EVALUATION_DATASET_PATH)
print("Recipes JSON path:", RECIPES_JSON_PATH)

Base directory: C:\Users\aless\Documents\cookmate-ai
Data directory: C:\Users\aless\Documents\cookmate-ai\data
Raw data directory: C:\Users\aless\Documents\cookmate-ai\data\raw
Processed data directory: C:\Users\aless\Documents\cookmate-ai\data\processed
Database directory: C:\Users\aless\Documents\cookmate-ai\db
Evaluation dataset path: C:\Users\aless\Documents\cookmate-ai\src\cookmate\monitoring\evaluation_dataset.json
Recipes JSON path: C:\Users\aless\Documents\cookmate-ai\data\processed\recipes_clean.json


## Open the recipe vector database

In [17]:
import lancedb

db = lancedb.connect(DB_DIR)

table_name = db.list_tables()
table_name


ListTablesResponse(tables=['recipes'], page_token=None)

## Open the recipes tabel

In [18]:
recipes_table = db.open_table("recipes")
recipes_table

LanceTable(name='recipes', version=2, _conn=LanceDBConnection(uri='C:\\Users\\aless\\Documents\\cookmate-ai\\db'))

## Inspect the recipes table

In [19]:
recipes_df = recipes_table.to_pandas()

print("NUmmer of records:", len(recipes_df))
print("Columns:", recipes_df.columns.to_list())

recipes_df.head()

NUmmer of records: 2870
Columns: ['id', 'text', 'vector']


,id,text,vector
0,recipe_0,Recipe: Pork Chops And Scalloped Potatoes. Ing...,"[0.0154290665, 0.016072212, -0.036934018, -0.0..."
1,recipe_1,Recipe: Hash Brown Potato Casserole. Ingredien...,"[0.09475726, 0.0187131, -0.051905304, -0.09564..."
2,recipe_2,Recipe: Ham And Potato Casserole. Ingredients:...,"[0.06773184, 0.03844117, -0.0122497175, -0.031..."
3,recipe_3,Recipe: Creamy Potato Casserole. Ingredients: ...,"[0.08041735, -0.060613405, -0.04692873, -0.057..."
4,recipe_4,Recipe: Quick Cheesy Potatoes(Microwave) . In...,"[0.022172667, -0.013191633, -0.055509403, -0.0..."


## Load the cleaned recipe data

The LanceDB table contains recipe ids, text, and vectors. We also load the cleaned JSON dataset so that we can map retrieved recipe ids back to structured recipe fields such as title, ingredients, and instructions.

In [20]:
with open(RECIPES_JSON_PATH, "r", encoding="utf-8") as f:
    recipes = json.load(f)

recipes_lookup = {recipe["id"]: recipe for recipe in recipes}

print("Number of recipes in JSON:", len(recipes))
print("Example recipe id:", recipes[0]["id"])
print("Example recipe title:", recipes[0]["title"])
print("Example recipe keys:", recipes[0].keys())

Number of recipes in JSON: 2870
Example recipe id: recipe_0
Example recipe title: Pork Chops And Scalloped Potatoes
Example recipe keys: dict_keys(['id', 'title', 'ingredients', 'instructions', 'text'])


## Test vector search with a sample ingredient query

This is a temporary retrieval test while the backend agent is still under development.

We create an embedding from a sample ingredient query and use it to retrieve the top 3 most similar recipes from the LanceDB table. This verifies that the vector database works correctly before connecting it to the final recipe agent.

In [21]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

query_ingredients = ["potatoes"]
query_text = ", ".join(query_ingredients)

query_vector = embedding_model.encode(query_text).tolist()

retrieved_rows = (
    recipes_table
    .search(query_vector)
    .limit(3)
    .to_pandas()
)

retrieved_rows[["id", "text"]]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1419.28it/s]


,id,text
0,recipe_2327,Recipe: Panfried Smashed Potatoes . Ingredient...
1,recipe_2032,Recipe: Mashed Red-Skinned Potatoes . Ingredie...
2,recipe_2827,"Recipe: Baked Potato. Ingredients: potato, but..."


## What are we doing:

## Combine LanceDB retrieval with structured recipe data

In the final application, recipe retrieval will be performed through the LanceDB vector database. This is the component that powers the RAG workflow and returns the most relevant recipe ids based on semantic similarity.

At the moment, the LanceDB table only stores three fields:

- `id`
- `text`
- `vector`

These fields are enough for retrieval, but they are not ideal for evaluation because we also want structured recipe fields that are easier to inspect and judge.

For evaluation, the most useful structured fields from `recipes_clean.json` are:

- `title`: the recipe name
- `ingredients`: the ingredients required by the recipe
- `instructions`: the cooking steps
- `text`: the full recipe text used for embeddings

To support evaluation, we combine two data sources:

- **LanceDB (`recipes` table)** for real semantic retrieval
- **`recipes_clean.json`** as a lookup table for structured recipe metadata

The retrieved recipe ids from LanceDB are used to fetch the full recipe information from the JSON file.

This allows us to test a realistic retrieval pipeline while still preparing structured outputs that are easier to inspect and evaluate with MLflow.

In the MLflow evaluation format, these structured recipe fields will mainly be used to build the `outputs` column. The `expectations` column will describe what a good answer should satisfy, such as using relevant recipes, including recipe titles, and avoiding invented recipes outside the database.